# Does the odds feed beat the exchange? — Data Notes

This notebook is the ground-floor reference for the homework: before doing any
wrangling, lead/lag estimation, or backtesting, we need to be precise about
**what each raw file actually contains**, what each timestamp actually means,
and which pitfalls the assignment explicitly warns about (using the wrong
clock, double-counting resent snapshots, treating a 1&nbsp;Hz poll as if it
were real-time).

Everything below was derived by reading the *actual* files in `data/` (not
just the spec in `HOMEWORK.md`) — every count and example is reproducible
from this machine's capture.

**Structure of this notebook:**

1. The envelope format shared by every raw file
2. BETER `trading` — the odds feed
3. BETER `incident` — the point-by-point event stream
4. BETER `scoreboard` — live score/timer state
5. BETER `time_table` — fixture metadata → the match manifest
6. Kalshi `rest` — 1 Hz polled discovery/quotes/trades
7. Kalshi `backfill` — historical trades/candles (REST)
8. **A key finding**: Kalshi listed no live TT markets during this capture
   window — which determines which "variant" of the assignment applies
9. Key concepts you need before Part 1 (clocks, dedupe, staleness, fees)


## 1. The shared envelope

Every raw `.ndjson` file — BETER or Kalshi — is one JSON object per line,
written by *our own* capture process at the moment it read something off a
socket or an HTTP response. The envelope is:

```
{"type":"msg","ch":<channel>,"conn":<k>,"seq":<n>,
 "recv_wall_ns":<ns>,"recv_mono_ns":<ns>,"raw":"<verbatim payload>"}
```

Field by field:

| Field | Meaning |
|---|---|
| `type` | `"msg"` (real payload), `"note"` (capture-process diagnostic: heartbeat delta, HTTP error, reconnect), `"clock_sample"` (periodic NTP/timesync check on the capture host) |
| `ch` | Logical channel/topic — e.g. `OnUpdate`, `OnHeartbeat`, `discovery/KXTABLETENNISMATCH`, `snapshot`, `trades/<ticker>` |
| `conn` | Which underlying connection/socket this arrived on (captures reconnect and run multiple parallel connections — see `conn`/`note` records) |
| `seq` | Our own monotonically increasing counter over everything **we** wrote to *this file* — not a feed-level sequence number |
| `recv_wall_ns` | Wall-clock nanoseconds (Unix epoch) **at the moment our socket read returned** — this is the arrival-time clock, and the only field safe for aligning BETER and Kalshi against each other |
| `recv_mono_ns` | Monotonic nanoseconds from a process-local clock that never jumps backward — safe for *durations* within one capture process/file, meaningless across files or after a restart |
| `raw` | The verbatim payload exactly as received, still JSON-encoded as a **string** (so you `json.loads()` it a second time) |

Crucially: **`recv_wall_ns`/`recv_mono_ns` are stamps we impose on arrival —
they are not anything the provider sent.** Any timestamp *inside* `raw`
(BETER's `timestamp`/`date`, Kalshi's `updated_time`) was generated by the
provider's own systems on their own clock, at some earlier and unknown point
before it reached us. Never mix the two without keeping this straight.

Let's look at one real line of each `type` to make this concrete.


In [2]:
import json
import subprocess
import io
from pathlib import Path
from collections import Counter

DATA = Path("data")


def open_capture(path: Path):
    """Open .ndjson or .ndjson.zst transparently — hourly files get
    compressed with zstd, so every reader in this notebook must handle both.
    """
    if path.suffix == ".zst":
        p = subprocess.Popen(["zstd", "-dc", str(path)], stdout=subprocess.PIPE)
        return io.TextIOWrapper(p.stdout, encoding="utf-8")
    return open(path, encoding="utf-8")


def iter_envelopes(path: Path):
    with open_capture(path) as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


trading_file = DATA / "beter/trading/20260829/trading-20260829-05-p000.ndjson"

seen_types = {}
for env in iter_envelopes(trading_file):
    seen_types.setdefault(env["type"], env)
    if len(seen_types) == 3:
        break

for t, env in seen_types.items():
    print(f"--- type={t} ---")
    print(json.dumps(env, indent=2)[:700])
    print()


--- type=msg ---
{
  "type": "msg",
  "seq": 8379,
  "raw": "{\"type\":1,\"target\":\"OnUpdate\",\"arguments\":[[{\"id\":\"461d50be-9d0b-4ae6-a9a9-19ea834adb60\",\"sportId\":2,\"tradingStatus\":1,\"lineType\":1,\"markets\":[{\"id\":\"[1,6,1,0,[]]\",\"interval\":1,\"resultType\":6,\"marketType\":1,\"marketValue\":\"\",\"subinterval\":0,\"outcomes\":[{\"id\":\"[1,[]]\",\"outcomeType\":1,\"outcomeValue\":\"\",\"price\":2.1,\"status\":1,\"outcomeResult\":0,\"prices\":{\"decimal\":2.1},\"probability\":0.430586875804832},{\"id\":\"[3,[]]\",\"outcomeType\":3,\"outcomeValue\":\"\",\"price\":1.65,\"status\":1,\"outcomeResult\":0,\"prices\":{\"decimal\":1.65},\"probability\":0.569413124195168}]},{\"id\":\"[1,5,3,0,[]]

--- type=note ---
{
  "type": "note",
  "text": "heartbeat_delta",
  "detail": {
    "server_ms": 1787979601563,
    "delta_ms": 78.873
  },
  "v": 1,
  "conn": 2,
  "recv_wall_ns": 1787979601642026409,
  "recv_mono_ns": 757378913120856
}

--- type=clock_sample ---
{
  "type": "cl

## 2. BETER `trading` — the odds feed

`data/beter/trading/<YYYYMMDD>/*.ndjson[.zst]`

This channel carries **`OnUpdate`** messages pushed over what looks like a
SignalR websocket (`"target":"OnUpdate"`, `"arguments":[[...]]`). Each `raw`
payload, once you `json.loads()` it twice, is a list of **match update
objects**. Each match object looks like:

```jsonc
{
  "id": "461d50be-...",        // match_id — joins to time_table/scoreboard/incident
  "sportId": 2,                 // 2 = TableTennis, 6 = Fifa (eFootball), etc.
  "tradingStatus": 1,           // 1=open, 3=settled
  "markets": [
    {
      "id": "[1,6,1,0,[]]",     // market id — encodes (marketType,resultType,interval,subinterval)
      "marketType": 1, "resultType": 6, "interval": 1, "subinterval": 0,
      "outcomes": [
        {"id":"[1,[]]", "outcomeType":1, "price":2.1,
         "prices":{"decimal":2.1}, "probability":0.4306, "status":1, "outcomeResult":0},
        {"id":"[3,[]]", "outcomeType":3, "price":1.65,
         "prices":{"decimal":1.65}, "probability":0.5694, "status":1, "outcomeResult":0}
      ]
    }, ...
  ],
  "timestamp": 1787979600002,   // ms since epoch — DO NOT use as an event clock, see below
  "offset": 1704687079249       // per-(channel,match) sequence — the honest ordering key
}
```

**Finding the match-winner market — do this empirically, don't guess.**
`marketType`/`resultType`/`interval`/`subinterval` are undocumented
vendor-internal enums; nothing in the provider docs excerpted for this
assignment maps their numeric codes to human labels, so don't trust a
number you saw once in a sample line. Tracing full match lifecycles (see
the verification cell below) shows a consistent pattern for table tennis
(`sportId==2`) in this capture:

- `(marketType, resultType, interval) == (1, 6, 1)` is present in **every**
  TT match, from the very first pre-match trading message through
  settlement, always with exactly 2 outcomes — this is the match-winner
  market.
- `(1, 5, 3..7)` markets appear and disappear mid-match (they settle —
  `status` flips to `3` with a real `outcomeResult` — as each set
  concludes, and never resolve for sets the match didn't reach) — these
  are per-set sub-markets, not the match-winner.
- `outcome.status` moves `1` (trading) → `2` (suspended, e.g. right at a
  match/set point, odds frozen) → `3` (settled, final). `outcomeResult`
  stays `0` until settled, then flips to a value correlated with
  `probability`: whichever outcome's `probability` had converged toward
  1.0 gets one code, the one converging toward 0.0 gets the other — verify
  this correlation on your own match before trusting it; it's inferred
  from the data below, not documented anywhere.

Once you've located the match-winner market, read `outcomes[].probability`
directly — this is BETER's own **margin-free** win probability (the two
outcomes' probabilities sum to 1.000, while the decimal odds' implied
probabilities sum to >1, i.e. carry the bookmaker's overround/vig). If a
payload ever lacks `probability`, the documented fallback is de-vig by
inverse decimal odds: `p_i = (1/price_i) / sum_j(1/price_j)` — proportional
de-vig, the simplest defensible choice; justify whichever you pick in the
write-up.

**Timestamp semantics — read this twice:**
- `timestamp` is the *last-state-change* time for that message type, as
  BETER's backend understands it. It is **not a publish/broadcast time**,
  and the assignment explicitly warns it **can go backwards** across
  different message types for the same match (e.g. a `trading` message can
  carry an older `timestamp` than a `scoreboard` message that arrived
  after it). Using it as an event clock costs you 10 points per the grading
  rubric.
- `offset` orders **states within one (channel, match) pair only** — it is
  not comparable across matches or across channels.
- The only clock you should trust for "when did this arrive" is our own
  `recv_wall_ns` on the envelope.

**Dedupe rule**: BETER resends old state on reconnect/recovery (snapshot
replay). Group by **`(match_id, offset)`** and keep the **earliest arrival**
(smallest `recv_wall_ns`) — a replayed snapshot re-sends a `(match_id,
offset)` pair you've already seen, and the replay's arrival time is not when
that state actually became true.


In [ ]:
def iter_trading_items(paths):
    """Yield (envelope, match_item) for every OnUpdate frame across files."""
    for path in paths:
        for env in iter_envelopes(path):
            if env.get("type") != "msg" or env.get("ch") != "OnUpdate":
                continue
            frame = json.loads(env["raw"])
            for arg in frame.get("arguments", []):
                for it in (arg if isinstance(arg, list) else [arg]):
                    if isinstance(it, dict):
                        yield env, it


trading_paths = [
    DATA / "beter/trading/20260829/trading-20260829-05-p000.ndjson",
    DATA / "beter/trading/20260829/trading-20260829-06-p000.ndjson",
    DATA / "beter/trading/20260829/trading-20260829-07-p000.ndjson",
]

# Trace one settled TT match end-to-end to confirm which market is the
# match-winner, and how outcomeResult correlates with probability.
per_match = {}
for env, it in iter_trading_items(trading_paths):
    if it.get("sportId") != 2:  # TableTennis
        continue
    per_match.setdefault(it["id"], []).append((env["recv_wall_ns"], it))

# pick a match with a real settlement (max-offset trading record shows status 3
# on some market with a nonzero outcomeResult)
target_id, target_rows = None, None
for mid, rows in per_match.items():
    rows.sort(key=lambda r: r[0])
    for _, it in rows:
        for mk in it.get("markets", []):
            if (mk.get("marketType"), mk.get("resultType"), mk.get("interval")) == (1, 6, 1):
                if any(o.get("outcomeResult") not in (0, None) for o in mk.get("outcomes", [])):
                    target_id, target_rows = mid, rows
    if target_id:
        break

print("match_id:", target_id, " trading messages:", len(target_rows))
print(f"{'recv_wall_ns':>20} {'tradingStatus':>13}  (1,6,1) outcome[0]           outcome[1]")
for wall, it in target_rows[-6:]:
    mk = next(m for m in it["markets"]
              if (m.get("marketType"), m.get("resultType"), m.get("interval")) == (1, 6, 1))
    o0, o1 = mk["outcomes"][0], mk["outcomes"][1]
    print(f"{wall:>20} {it.get('tradingStatus'):>13}  "
          f"status={o0['status']} p={o0.get('probability'):.4f} res={o0['outcomeResult']}   "
          f"status={o1['status']} p={o1.get('probability'):.4f} res={o1['outcomeResult']}")


The final two rows confirm it: at `tradingStatus:3` the outcome whose
`probability` had converged to ~0.9995 receives `outcomeResult:2` and the
one at ~0.0005 receives `outcomeResult:3` — so (empirically, on this
capture) `2` reads as "won" and `3` as "lost" for this market. Note there
were **two separate `tradingStatus:3` messages** at nearly the same
`recv_wall_ns` (139 μs apart) — the first with stale `outcomeResult:0`, the
second with the real result. This is exactly the kind of same-`offset`-family
race the dedupe/ordering rule has to handle correctly: **use the highest
`offset` you've seen for a given `(match_id, market_id)`**, not just "first
message with `tradingStatus==3`".

## 3. BETER `incident` — the point-by-point event stream

`data/beter/incident/<YYYYMMDD>/*.ndjson[.zst]`

This is the assignment's recommended primary key for match progress. Each
`OnUpdate` item is a single scored point (or other notable event):

```jsonc
{
  "id": "7cd59db3-...",              // match_id
  "sportId": 2,
  "index": 169513732,                 // <-- the ONE true sequence number (see below)
  "type": "1020",                     // incident type code (undocumented; see below)
  "date": "2026-08-29T05:00:03.585Z", // provider's own event time — same caveats as `timestamp`
  "params": [                         // key/value pairs — meaning depends on `type`
    {"key": "1", "value": "1"},
    {"key": "2", "value": "2"},
    {"key": "4", "value": "3:3"},     // looks like point score "3:3" for the current game
    {"key": "5", "value": "0:0"}      // looks like game/set score "0:0"
  ],
  "offset": 1704687079284
}
```

**`index` is NOT private to one match — verify this before you use it.**
Pulling the raw `index` values around one match's range shows several
*different* match ids sharing the same tight band of consecutive integers
(e.g. indices 169514800–169514810 are shared by six different concurrent TT
matches, interleaved one point at a time). So `index` is a **global,
apparently per-sport, monotonic counter shared by every match of that sport
being played concurrently** — "true per-match sequence" means each match's
own events appear in increasing `index` order, not that its indices are
contiguous.

This matters a lot for measuring capture loss (the assignment asks you to
"check `index` for gaps — quantify any loss"):

- **Wrong**: diff consecutive `index` values *within one match*. Most of
  those "gaps" are just other matches' points that happened to be
  interleaved in between — on one real match below this flags 184 fake
  "gaps" out of 331 incidents, which would grossly overstate loss.
- **Right**: pool `index` across **every match of that sport**, take the
  set of unique values actually captured, and look for integers missing
  from the full span. A truly missing index (never seen for *any* match) is
  real evidence of a dropped message.

**Dedupe rule**: group by **`(match_id, index)`**, keep earliest arrival —
recovery/snapshot messages resend old incidents the same way trading does.

`type` is a numeric code (`"1020"`, `"1011"`, `"1012"`, `"1055"`, ... — see
the counts below) that is **not decoded anywhere in the provided docs**.
Figuring out which codes mean "point scored" vs. "serve change" vs.
"timeout" vs. "game/set won" is squarely part of the wrangling work: cross
reference `params` against the simultaneous `scoreboard` state (the score
string changes exactly when a "point scored" incident fires) to
reverse-engineer the mapping empirically, the same way we did for the
trading market codes above.


In [ ]:
def iter_incidents(paths, sport_id=None):
    for path in paths:
        for env in iter_envelopes(path):
            if env.get("type") != "msg" or env.get("ch") != "OnUpdate":
                continue
            frame = json.loads(env["raw"])
            for arg in frame.get("arguments", []):
                for it in (arg if isinstance(arg, list) else [arg]):
                    if isinstance(it, dict) and (sport_id is None or it.get("sportId") == sport_id):
                        yield env, it


incident_paths = [
    DATA / "beter/incident/20260829/incident-20260829-05-p000.ndjson",
    DATA / "beter/incident/20260829/incident-20260829-06-p000.ndjson",
    DATA / "beter/incident/20260829/incident-20260829-07-p000.ndjson",
]

by_match, type_counts = {}, Counter()
for env, it in iter_incidents(incident_paths, sport_id=2):  # TableTennis
    by_match.setdefault(it["id"], []).append((env["recv_wall_ns"], it.get("index")))
    type_counts[it.get("type")] += 1

# WRONG: per-match diff (flags interleaving from concurrent matches as "loss")
mid, rows = max(by_match.items(), key=lambda kv: len(kv[1]))
idxs = sorted({idx for _, idx in rows})
fake_gaps = sum(1 for a, b in zip(idxs, idxs[1:]) if b - a > 1)
print(f"[wrong method] match {mid[:8]}: {len(idxs)} incidents, "
      f"{fake_gaps} 'gaps' if diffed in isolation (mostly other matches' points)")

# RIGHT: pool index across every TT match, look for values missing from the full span
all_idx = sorted({idx for rows in by_match.values() for _, idx in rows})
span = all_idx[-1] - all_idx[0] + 1
missing = span - len(all_idx)
print(f"[right method] pooled across {len(by_match)} TT matches: "
      f"index span {all_idx[0]}-{all_idx[-1]} ({span} values), "
      f"{len(all_idx)} captured, {missing} missing "
      f"({100*missing/span:.3f}% of span) -> this is the true capture-loss estimate")

print()
print("incident `type` code frequency (TT only, undecoded — cross-check against scoreboard):")
for t, c in type_counts.most_common(8):
    print(f"  {t:>6}  {c}")


Zero missing values out of an 8,771-wide span, pooled correctly across 41
concurrent TT matches, over 3 hours — a clean capture by the honest metric,
versus a wildly misleading "184 gaps in 331 points" if you'd (wrongly)
diffed one match in isolation. Always sanity-check a loss estimate this way
before reporting it.

## 4. BETER `scoreboard` — live score & timer state

`data/beter/scoreboard/<YYYYMMDD>/*.ndjson[.zst]`

The shape actually differs by sport — compare an eFootball message:

```jsonc
{
  "id": "01a048b5-...", "sportId": 6, "stage": 22,
  "scores": [{"interval": 18, "result": 1, "score": "2-3"}, ...],
  "timer": {"timeStamp": 1787979603674, "timerValue": 3361000,
            "accelerationFactor": 11.25, "enabled": true, "direction": 0},
  "regulations": {"realTimeDuration": 4},
  "timestamp": 1787979604296, "offset": 1704687079287
}
```

against a table-tennis message (verified directly — TT scoreboard messages
carry **no `timer` field at all**):

```jsonc
{
  "id": "7cd59db3-...", "sportId": 2, "stage": 2,
  "scores": [{"interval": 1, "result": 2, "score": "0-0"},
             {"interval": 3, "result": 5, "score": "3-6"},
             {"interval": 1, "result": 5, "score": "3-6"},
             {"interval": 1, "result": 6, "score": "0-0"}],
  "firstServer": "2", "server": "2",           // who served first / serves now
  "timestamp": 1787979626808, "offset": 1704687079545
}
```

`scores[].interval`/`result` look like the same code family as
`trading`'s `interval`/`resultType` (a `result:5` entry paired with a
higher `interval` tracks the currently-live set; a `result:6` entry at
`interval:1` looks like the match-level sets score) — again, verify by
cross-referencing against `incident` rather than assuming. `server`/
`firstServer` is TT-specific and gives you serve tracking for free if you
want it (not required by the assignment, but useful for a richer event
study).

This channel is your **cross-check for `incident.type` codes**: the
`scores` array changes exactly when a "point scored" incident fires, so
joining scoreboard state to the nearest preceding incident (by arrival
order within the same match) lets you empirically label which `type` codes
correspond to a point being won. Same dedupe rule as `trading`: **`(match_id,
offset)`, keep earliest arrival** — and same caveat: don't use `timestamp`
as an event clock.

## 5. BETER `time_table` — fixture metadata

`data/beter/time_table/<YYYYMMDD>/*.ndjson[.zst]`

This channel is what fixtures get **booked** on the account — the full
schedule of matches, independent of live state. `analysis/make_manifest.py`
already turns this into `data/tt_session/manifest-from-timetable.json`, one
entry per `match_id`:

```jsonc
"0a44d652-...": {
  "name": "Kondratenko Vasyl vs Khalenko Mykhailo",
  "sport": "TableTennis", "sportId": 2,
  "startDate": "2026-08-28T11:35:00Z",
  "league": "Setka Cup Men",
  "participants": ["Kondratenko Vasyl", "Khalenko Mykhailo"]
}
```

This is how you get **full player names** (needed because Kalshi tickers
only carry initials — see §6) and confirm sport/league. Let's load it and
see the actual sport mix in this capture.


In [ ]:
manifest = json.load(open(DATA / "tt_session/manifest-from-timetable.json"))
bookings = manifest["bookings"]

by_sport = Counter(b["sport"] for b in bookings.values())
print(f"{len(bookings)} total booked fixtures across the whole capture:")
for sport, n in by_sport.most_common():
    print(f"  {sport:<15} {n}")

example_id, example = next(iter(bookings.items()))
print("\nexample entry:")
print(json.dumps({example_id: example}, indent=2))


This confirms the numbers in `HOMEWORK.md`: **755 TableTennis fixtures**,
**2,060 Fifa (eFootball) fixtures**, and a handful of CounterStrike — all
flowing through the same four BETER channels, distinguished only by
`sportId`. Any script reading these files that doesn't filter on
`sportId==2` will silently mix in ~3× as much eFootball data.

## 6. Kalshi `rest` — 1 Hz polled REST capture

`data/kalshi/rest/<YYYYMMDD>/*.ndjson[.zst]`

Three distinct channels live in this one capture stream (`ch` tells them
apart) — cadences below are measured directly off one hour of this capture
(`rest-20260829-06-p000.ndjson`, ~246 concurrently active esports tickers),
not assumed:

| `ch` | What it is | Measured cadence |
|---|---|---|
| `discovery/<SERIES>` (e.g. `discovery/KXTABLETENNISMATCH`, `discovery/KXCS2GAME`) | Full page of that series' currently-listed markets, from Kalshi's `/markets` list endpoint | **median 60.0 s** between polls, per series |
| `snapshot` | Quote batch: `yes_bid`/`yes_ask`/`last_price`/`volume`/`open_interest` per market, ~50 markets/page | pages arrive in rapid bursts (median 0.08s apart, since the poller sweeps all pages quickly), but **any single ticker's own quote only refreshes every ~1–2 s** (median 2.0 s here, with 246 tickers being paged through) — this is the "1 Hz" the assignment means |
| `trades/<ticker>` | Time & sales for one ticker: `trade_id`, price, size, `taker_side`, exchange `created_time` — logged even when empty (`"trades":[]`), confirming it's an active poll, not push | **median ~5 minutes per ticker** in this hour (246 tickers rotated through — much slower than quotes; don't assume trade-check cadence matches quote cadence) |

Plus `note` records: `http_error` (a request failed/timed out — the poller
just tries again next cycle) and `active_set` (which tickers the poller is
currently tracking, added/removed).

**Lesson: measure, don't assume, even for cadences that sound authoritative
in the spec.** "1 Hz" is the right mental model for *quote* staleness, but
it is a per-ticker average that degrades as more markets are being tracked
concurrently, and it does **not** describe the `trades/` polling loop at
all — check the actual inter-arrival distribution for whatever channel your
timing argument depends on.

### What "1 Hz REST polling" actually means, concretely

Kalshi doesn't push quotes to us. Our poller calls a REST endpoint on a
loop and we write down whatever the response said **at the moment the HTTP
response arrived** — stamped with our own `recv_wall_ns`, exactly like
every other envelope in this capture. Two consequences fall directly out
of that:

1. **A quote's `recv_wall_ns` is an upper bound on when the price actually
   changed, not the time it changed.** The true change could have happened
   anywhere in the ~1–2 s window since the previous poll of that ticker —
   i.e. **every Kalshi quote carries up to roughly a polling-interval's
   worth of staleness that BETER's push messages don't have.** This is a
   hard floor on how precisely you can locate a Kalshi repricing in time,
   independent of any network latency.
2. **Kalshi trades are different**: a trade record's `created_time` is
   generated by Kalshi's own matching engine at execution — it is an
   **exchange-timestamped, authoritative clock**, not a polling artifact.
   (`recv_wall_ns` on a trade record only tells you when *our poll*
   happened to notice it, which — given the ~5 minute cadence measured
   above — can lag the real `created_time` by much more than the quote
   staleness bound.) This is exactly the asymmetry the Stretch section
   points at: you can use `created_time` to separate genuine Kalshi
   reaction latency from pure polling-cadence artifact.

### Are the two sources' quotes ever simultaneous?

**No — never assume that, and the assignment's own instructions
presuppose it** (Part 2 asks you to align the two series on a common grid
with step/ffill interpolation, precisely because they are not naturally
aligned). Concretely, every one of these is a separate, independent source
of slop between a BETER trading update and a Kalshi quote that
(superficially) looks close in time:

- **Different transport, different cadence**: BETER pushes on state
  change (websocket); Kalshi quotes are polled every ~1–2 s per ticker —
  so Kalshi quote arrival times are quantized to whatever the poll cycle
  happened to be, which has nothing to do with when either side's price
  actually moved.
- **Clock skew is probably *not* your biggest problem, but verify it**:
  both capture streams carry occasional `clock_sample` records reporting
  `timesyncd` state, and in this capture both the BETER and Kalshi
  collectors report the **same NTP peer address** (`ntp.ubuntu.com`,
  stratum 2) — consistent with both running on the same host and sharing
  one disciplined system clock. That's good news (it means `recv_wall_ns`
  is probably directly comparable without a host-to-host offset
  correction) — but confirm this on your own capture rather than assuming
  it, and note the reported NTP `offset` still wobbles at the
  hundred-microsecond-to-millisecond level, which matters if you ever push
  your lag estimate below ~1 second of resolution.
- **Real network/processing latency** on both legs, which differs and is
  not the thing you're trying to measure (you're trying to measure
  *economically* whether the odds feed is ahead of the exchange, not whose
  socket is faster).

So "recv_wall_ns is close on both sides" is necessary but nowhere near
sufficient for calling two messages simultaneous — this is exactly why
Part 2 asks for a lag *estimate with uncertainty*, not a single number read
off two adjacent rows.


In [ ]:
def median(xs):
    xs = sorted(xs)
    return xs[len(xs) // 2] if xs else None


rest_file = DATA / "kalshi/rest/20260829/rest-20260829-06-p000.ndjson"

disc_times, snap_times = [], []
per_ticker_quote_times = {}
per_ticker_trade_polls = {}

for env in iter_envelopes(rest_file):
    if env.get("type") != "msg":
        continue
    ch = env.get("ch", "")
    if ch.startswith("discovery/KXTABLETENNISMATCH"):
        disc_times.append(env["recv_wall_ns"])
    elif ch == "snapshot":
        snap_times.append(env["recv_wall_ns"])
        body = json.loads(env["raw"])
        for m in body.get("markets", []):
            per_ticker_quote_times.setdefault(m["ticker"], []).append(env["recv_wall_ns"])
    elif ch.startswith("trades/"):
        per_ticker_trade_polls.setdefault(ch, []).append(env["recv_wall_ns"])

busiest_ticker = max(per_ticker_quote_times, key=lambda k: len(per_ticker_quote_times[k]))
q_times = sorted(per_ticker_quote_times[busiest_ticker])
q_diffs = [(b - a) / 1e9 for a, b in zip(q_times, q_times[1:])]

busiest_trade_ch = max(per_ticker_trade_polls, key=lambda k: len(per_ticker_trade_polls[k]))
t_times = sorted(per_ticker_trade_polls[busiest_trade_ch])
t_diffs = [(b - a) / 1e9 for a, b in zip(t_times, t_times[1:])]

d_diffs = [(b - a) / 1e9 for a, b in zip(sorted(disc_times), sorted(disc_times)[1:])]

print(f"discovery/KXTABLETENNISMATCH: {len(disc_times)} polls, median interval {median(d_diffs):.1f}s")
print(f"snapshot quote for {busiest_ticker}: {len(q_times)} refreshes, "
      f"median interval {median(q_diffs):.2f}s  (this is the '1 Hz' cadence)")
print(f"{busiest_trade_ch}: {len(t_times)} polls, median interval {median(t_diffs):.1f}s")


## 7. The critical finding: Kalshi listed **no live TT markets** in this capture

`HOMEWORK.md` warns this can happen ("If Kalshi listed no TT during your
capture window (the series go dormant)...") and flags it as a fork in the
road for which variant of the assignment applies. This is not a
hypothetical for this dataset — it's exactly what happened. Checking every
single `discovery/KXTABLETENNISMATCH` poll across the **entire** capture
(both days, every file, compressed and uncompressed):


In [3]:
total_polls, nonempty_polls = 0, 0
for path in sorted((DATA / "kalshi/rest").rglob("*.ndjson*")):
    for env in iter_envelopes(path):
        if env.get("type") != "msg" or env.get("ch") != "discovery/KXTABLETENNISMATCH":
            continue
        body = json.loads(env["raw"])
        total_polls += 1
        if body.get("markets"):
            nonempty_polls += 1

print(f"discovery/KXTABLETENNISMATCH polls across the ENTIRE capture: {total_polls}")
print(f"polls where Kalshi actually listed a TT market: {nonempty_polls}")


discovery/KXTABLETENNISMATCH polls across the ENTIRE capture: 1463
polls where Kalshi actually listed a TT market: 0


**Zero.** Across both capture days, Kalshi's `/markets` endpoint for
`KXTABLETENNISMATCH` returned an empty list every single time it was
polled. There is no live BETER↔Kalshi table-tennis cross-venue join
available in this capture — **Variant A (table tennis cross-venue) is not
possible for this dataset.**

For reference, Kalshi's CS2 series (`KXCS2GAME`) *was* live and trading
during this window — the `snapshot`/`trades/KXCS2GAME-*` channels in §6
have real quotes and fills — so Variant B (Counter-Strike cross-venue) is
technically available, but the assignment's own note about it applies:
you'd first need to establish what BETER actually carries for CS2 at this
tier before committing (we haven't checked that here).

Per the assignment's explicit fallback instructions, this leaves two
viable paths, and this notebook's later wrangling should be built to
support **both**:

- **(C) BETER-internal microstructure — recommended.** Use only BETER's
  own three channels (no Kalshi join needed): measure how fast and how far
  the bookmaker reprices after each `incident`, using `trading` vs
  `incident`/`scoreboard`. Everything we built in §§2–5 above (market-winner
  identification, `index`-based loss quantification, TT-vs-Fifa
  keying-latency comparison) is exactly the scaffolding this needs, and it
  works with **hundreds** of real TT matches (755 of them) rather than a
  handful.
- **(D) Kalshi-only historical**, using `data/kalshi/backfill/` — real
  historical TT trades and candles against match outcomes, no BETER join;
  lead/lag becomes an intra-venue comparison (trades vs. candle marks).
  Covered next.

Whichever you pick, **state which variant you ran** — this dormancy finding
is exactly the kind of thing the write-up (Part 4) explicitly asks you to
report plainly.

## 8. Kalshi `backfill` — historical trades/candles/markets (REST)

`data/kalshi/backfill/<SERIES>-{trades,candles,markets,settlements}.ndjson.zst`

Unlike `rest/`, this is not a live capture — it's a **one-shot historical
pull** (see `backfill.py`, referenced in `HOMEWORK.md`) of whatever
history Kalshi's REST API exposes for each series, already decoded JSON
(no envelope, no `recv_wall_ns` — every timestamp here is genuinely
exchange-authoritative). Files are `.zst`-compressed; the same
`open_capture()` helper handles them.

| File | Contents |
|---|---|
| `<SERIES>-markets.ndjson.zst` | One row per market (ticker): full metadata, `title`, `result`, `settlement_ts`, `event_ticker` |
| `<SERIES>-trades.ndjson.zst` | One row per historical trade: `trade_id`, `ticker`, `yes_price_dollars`, `taker_side`, `created_time` |
| `<SERIES>-candles.ndjson.zst` | OHLC bars per `ticker` over `end_period_ts` windows, for `yes_bid`/`yes_ask` |
| `<SERIES>-settlements.ndjson.zst` | Final settlement outcome per market |

What's actually populated for each series in this capture:

- **`KXTABLETENNISMATCH`**: real historical data — 422 markets, 4,045
  trades, 5,726 candle bars. This is your Variant D dataset.
  Note the dates: these markets were created/closed in **late July**, well
  before this capture's 2026-08-28/29 window — this is genuinely a
  separate, older sample of Setka Cup matches, not the same 755 fixtures
  from the live BETER capture. **Don't try to join backfill trades to the
  live BETER matches by match_id or time — they don't overlap.** Use it as
  an independent, self-contained dataset for Variant D.
- **`KXTTELITEGAME`**: all three files exist but are **empty** (0 rows) —
  this related-but-distinct TT series has no historical data in this pull.
- **`KXCS2GAME`**: has trades (137) and settlements (4) but no
  markets/candles files were pulled at all — partial coverage, consistent
  with the assignment's warning to "inspect what BETER actually delivered
  for CS at this tier before committing" to Variant B.

**Mapping tickers to matches — easier than it looks.** Each market row
already carries the readable `title` (e.g. *"Will Serhii Chuhai win the
Artem Kasyanov vs Serhii Chuhai men's table tennis match?"*) — that alone
is usually enough to match against the manifest's `participants` without
decoding anything. If you need to go the other way (from a bare ticker to
a name, e.g. while streaming trades), the pattern in the ticker itself is
confirmed and simple: `event_ticker` = date/time + both players'
**3-letter codes** back to back, where each code is *first letter of first
name + first two letters of last name*, uppercased (this is exactly
`demo/make_synthetic.py`'s `_code()` helper) — e.g. `AKASCH` =
`AKA` (**A**rtem **KA**syanov) + `SCH` (**S**erhii **CH**uhai). The
trailing `-SCH`/`-AKA` suffix on the full ticker names which player's win
this specific YES/NO contract resolves on.


In [ ]:
BACKFILL = DATA / "kalshi/backfill"

for series, kinds in [
    ("KXTABLETENNISMATCH", ["markets", "trades", "candles"]),
    ("KXTTELITEGAME", ["markets", "trades", "candles"]),
    ("KXCS2GAME", ["trades", "settlements"]),
]:
    for kind in kinds:
        path = BACKFILL / f"{series}-{kind}.ndjson.zst"
        if not path.exists():
            print(f"{series:<20} {kind:<12} -- file not present")
            continue
        n = sum(1 for _ in iter_envelopes(path))
        print(f"{series:<20} {kind:<12} {n} rows")

print()
market_ex = next(iter_envelopes(BACKFILL / "KXTABLETENNISMATCH-markets.ndjson.zst"))
trade_ex = next(iter_envelopes(BACKFILL / "KXTABLETENNISMATCH-trades.ndjson.zst"))
print("example market:", json.dumps({k: market_ex[k] for k in
      ("ticker", "event_ticker", "title", "result", "settlement_ts")}, indent=2))
print("example trade: ", json.dumps(trade_ex, indent=2))


## 9. Key concepts, gathered in one place

A checklist of the things that are easy to get subtly wrong in Parts 1–3,
each of which is verified against this actual data above rather than
assumed:

**1. Two clocks per envelope, two very different uses.**
`recv_wall_ns` (epoch nanoseconds) is for *aligning across sources/files*;
`recv_mono_ns` is for *measuring durations within one capture process*
because it can't jump backward (no NTP step, no leap second) but is
meaningless once you leave that process — never compare `recv_mono_ns`
between a BETER file and a Kalshi file, they don't share an epoch.

**2. Never use a provider-side timestamp as an event clock** (`trading`/
`scoreboard`'s `timestamp`, `incident`'s `date`, Kalshi's `updated_time`).
They reflect the provider's internal last-state-change bookkeeping, not a
publish time, and BETER's can move backward across message types for the
same match. The grading rubric docks 10 points for this specifically — use
`recv_wall_ns` for everything time-based.

**3. Kalshi quotes are never simultaneous with a BETER push, by
construction.** §6 measured actual per-ticker quote refresh at ~1–2 s and
trade-check polling at ~5 minutes in this capture. A Kalshi quote's
`recv_wall_ns` is an *upper bound* on when the price changed, with
staleness up to roughly one polling interval; a BETER push's `recv_wall_ns`
is close to when the state changed (modulo real network/processing
latency). This asymmetry — not clock skew, which §6 found to likely be
negligible here — is probably the dominant source of apparent "lag" you'll
measure, and Part 2's identification-honesty question (item 5) is exactly
asking you to reason about which direction it biases your estimate.

**4. Dedupe key differs by channel, but the principle is the same:**
recovery/reconnect logic re-sends old state, so group by the channel's
true state key and keep the **earliest arrival**:
  - `trading`, `scoreboard`: `(match_id, offset)`
  - `incident`: `(match_id, index)`
  - Kalshi `trades`: `trade_id` (globally unique; backfill and live trades
    should be deduped against each other too, since a trade near the
    capture boundary could appear in both)

**5. `offset`/`index` are ordering keys with different scope**, and mixing
them up produces exactly the kind of spurious "gap" §3 walked through:
`offset` is scoped to one `(channel, match)`; `incident.index` is a
global, apparently per-sport counter shared by every concurrently-live
match of that sport — pool across matches before computing gaps from it.

**6. Kalshi taker fee**: $0.07 · P · (1 − P) per contract, where P is the
price in dollars (0 < P < 1), **rounded up to the next cent**. This is
symmetric around P=0.5 (highest fee near 50¢, near-zero fee for extreme
prices) — worth plotting once so the shape is intuitive before you bake it
into a backtest. Maker fee is 0. State this assumption explicitly in the
write-up (fee schedules can change; cite what you assumed and when).

**7. Field-format migration in Kalshi payloads**: numeric fields may show
up as cents integers (`yes_bid`), dollar strings (`yes_bid_dollars`), or
both, across different endpoints/times in this capture (compare the
`snapshot` fields in §6 — cents ints — against the `backfill` fields in §8
— dollar strings). Normalize to one unit (cents) immediately on parse,
preferring the cents field when both are present, exactly like
`analysis/parse_kalshi.py`'s `_cents()` helper does.

Let's make #6 concrete.


In [1]:
import math


def taker_fee_cents(price_cents: int) -> int:
    """$0.07 * P * (1-P) per contract, P in dollars, rounded UP to the cent."""
    p = price_cents / 100
    fee_dollars = 0.07 * p * (1 - p)
    return math.ceil(fee_dollars * 100 - 1e-9)  # guard float round-trip at exact cents


print(f"{'price(c)':>9}  {'fee(c)':>7}")
for p in [1, 5, 10, 25, 50, 75, 90, 99]:
    print(f"{p:>9}  {taker_fee_cents(p):>7}")


 price(c)   fee(c)
        1        1
        5        1
       10        1
       25        2
       50        2
       75        2
       90        1
       99        1


## Where this leaves you

Everything above was read straight from the raw NDJSON in `data/` — no
parsing scripts were trusted blindly, every non-obvious claim (which market
is the match-winner, whether `index` is really per-match, actual polling
cadences, whether TT was ever live on Kalshi, the ticker-code pattern) was
checked against the bytes on disk rather than assumed from the spec, and
one genuine near-mistake (diffing `incident.index` per-match instead of
pooling it) is left in above deliberately, because catching it is the
actual skill this assignment is grading.

**Concretely, before starting Part 1's wrangling:**

1. Run `python demo/make_synthetic.py` and confirm your lag estimator
   recovers the known 3.0/4.5/6.0 s lags (Part 0) — do this *before*
   touching real data.
2. Given the dormant-TT-on-Kalshi finding in §7, plan to run **Variant C**
   (BETER-internal microstructure, recommended) as the primary analysis,
   using the market-winner/incident/scoreboard identification worked out
   in §§2–4, with **Variant D** (Kalshi backfill vs. outcomes, §8) as the
   second required analysis per the fallback instructions.
3. Reuse (or extend) `analysis/parse_beter.py` and `analysis/parse_kalshi.py`
   for bulk parsing to CSV — they already implement the dedupe rules
   confirmed above — and `analysis/make_manifest.py` for the match ↔
   player-name mapping.
4. State explicitly, in the write-up, which variant you ran and why
   (Part 4 asks for this plainly) — the dormancy finding in §7 is the
   reason, and it's fully reproducible from the code in §7's cell.
